In [2]:
def f3d(figure, **kwargs):
    return make_frame3d(figure, **kwargs)
#@options(grid=(8,8,8), colors=['white', 'gray'], alpha=.4, resize_factor=1.2, show_axes=False, show_labels=True, label_names=['x','y','z'], show_ticks=True)
def make_frame3d(figure, grid=(8,8,8), colors=['white', 'gray'], alpha=.3, resize_factor=1.2, show_axes=False, show_labels=True, label_names=['x','y','z'], show_ticks=True, **kwargs):
   
    fig_boundings = tuple(zip(*figure.bounding_box()))
    lower_left = [ur - (ur - ll)*resize_factor for ll, ur in fig_boundings]
    upper_right = [ll + (ur - ll)*resize_factor for ll, ur in fig_boundings]
    frame_color, gridline_color = colors
    figure += set_frame3d(lower_left, upper_right, grid, frame_color, gridline_color, show_axes, show_labels, label_names, show_ticks, alpha=alpha, **kwargs)
    if 'aspect_ratio' not in kwargs.keys():
        dx, dy, dz = (ur - ll for ll, ur in fig_boundings)
        dmax = max(dx,dy,dz)
        ratio = [dmax/k for k in (dx,dy,dz)]
        figure.aspect_ratio(ratio)
    return figure
def set_frame3d(lower_left, upper_right, grid, frame_color, gridline_color, show_axes=True, show_labels=True, label_names=['x','y','z'], show_ticks=True, **kwargs):
    
    frame = set_background_frame(lower_left,upper_right, color=frame_color, frame=False, **kwargs)
    frame += set_frame_gridlines(lower_left,upper_right, grid=grid, color=gridline_color)
    if show_labels and not show_axes:
        frame += set_axes_labels(lower_left, upper_right, label_names, 
                              fontfamily='serif', fontweight='bold', fontsize='140%')
    if show_ticks:
        frame += set_frame_ticks(lower_left,upper_right,grid=grid, 
                              fontfamily='serif')
    if show_axes:
        frame += set_axes(lower_left, upper_right, label_names, fontfamily='serif', fontweight='bold', fontsize='140%')
    return frame
def set_axes(lower_left, upper_right, label_names, **kwargs):
    from sage.modules.free_module_element import vector
    from sage.plot.plot3d.shapes2 import text3d
    x_length,y_length,z_length = [(ur - ll)*6/5 for ll,ur in zip(lower_left,upper_right)]
    v0 = vector(lower_left)
    vx = vector([x_length,0,0])
    vy = vector([0,y_length,0])
    vz = vector([0,0,z_length])
    size = max(x_length,y_length,z_length)
    vectors = sum((v.plot(color='gray',frame=False,thickness=size*4/6, start=v0) for v in (vx,vy,vz)))
    labels = sum([text3d(l,1.1*v+v0,**kwargs) for l,v in zip(label_names,(vx,vy,vz))])
    return vectors+labels
def grid_slices(lower_left, upper_right, grid):    
    from sage.arith.srange import srange
    steps = ((b-a)/(c-1) for a,b,c in zip(lower_left, upper_right, grid))
    return (srange(a+c,b,c, include_endpoint=True) for a,b,c in zip(lower_left, upper_right, steps))
	def set_background_frame(lower_left, upper_right, **kwargs):
    from sage.plot.plot3d.shapes2 import polygon3d
    if 'theme' in kwargs.keys() and kwargs['theme'] == 'dark' and 'color' in kwargs.keys() and kwargs['color'] == 'white':
        kwargs['color']='black'
        kwargs['alpha']=.1
    xmin, ymin, zmin = lower_left
    xmax, ymax, zmax = upper_right
    xz_plane = polygon3d([lower_left, (xmin, ymin, zmax), (xmax, ymin, zmax),
                          (xmax, ymin, zmin), lower_left], **kwargs)
    xy_plane = polygon3d([lower_left, (xmin, ymax, zmin), (xmax, ymax, zmin),
                          (xmax, ymin, zmin), lower_left], **kwargs)
    yz_plane = polygon3d([lower_left, (xmin, ymax, zmin), (xmin, ymax, zmax),
                          (xmin, ymin, zmax), lower_left], **kwargs)
    planes = xz_plane + xy_plane + yz_plane
    return planes      
def set_frame_gridlines(lower_left, upper_right, grid, **kwargs):    
    from sage.plot.plot3d.shapes2 import line3d
    xmin, ymin, zmin = lower_left
    xmax, ymax, zmax = upper_right
    xcoords, ycoords, zcoords = grid_slices(lower_left, upper_right, grid)
    x_gridlines = sum([line3d([(xc, ymax, zmin), (xc, ymin, zmin),
                      (xc, ymin, zmax)], **kwargs) for xc in xcoords])
    y_gridlines = sum([line3d([(xmax, yc, zmin), (xmin, yc, zmin),
                      (xmin, yc, zmax)], **kwargs) for yc in ycoords])
    z_gridlines = sum([line3d([(xmax, ymin, zc), (xmin, ymin, zc),
                      (xmin, ymax, zc)], **kwargs) for zc in zcoords])
    border = line3d([(xmin, ymin, zmax), (xmax, ymin, zmax),(xmax, ymin, zmin),(xmax, ymax, zmin), 
                    (xmin, ymax, zmin), (xmin, ymax, zmax), (xmin, ymin, zmax)], thickness=2, **kwargs)
    return x_gridlines + y_gridlines + z_gridlines + border  
def set_axes_labels(lower_left, upper_right, label_names, **kwds):    
    from sage.plot.plot3d.shapes2 import text3d
    xmin, ymin, zmin = lower_left
    xmed, ymed, zmed = (a + (b - a)/2 for a, b in zip(lower_left,upper_right))
    x1, y1, z1 = (b + (b - a)/5 for a, b in zip(lower_left,upper_right))
    xlabel, ylabel, zlabel = label_names
    labels = text3d(xlabel, (xmed, y1, zmin), **kwds)
    labels += text3d(ylabel, (x1, ymed, zmin), **kwds)
    labels += text3d(zlabel, (x1, ymin, zmed), **kwds)
    return labels    
def set_frame_ticks(lower_left, upper_right, grid, eps=1, **kwargs):    
    from sage.plot.plot3d.shapes2 import text3d
    xmin, ymin, zmin = lower_left
    xmax, ymax, zmax = upper_right
    xcoords, ycoords, zcoords = grid_slices(lower_left, upper_right, grid)
    if eps == 1: eps = 1.2*eps
    x_ticks = sum([text3d(round(xc, 2),(xc, eps*ymax, zmin), 
                           **kwargs) for xc in xcoords])
    y_ticks = sum([text3d(round(yc, 2),(eps*xmax, yc, zmin), 
                           **kwargs) for yc in ycoords])
    z_ticks = sum([text3d(round(zc, 2),(eps*xmax, ymin, zc),
                           **kwargs) for zc in zcoords])
    return x_ticks + y_ticks + z_ticks
var('x y')
f3d(plot3d((x^2 + y^2),(x,-3,3),(y,-3,3), plot_points=200))

TabError: inconsistent use of tabs and spaces in indentation (<string>, line 46)

In [ ]:
#from sage.plot.plot3d.shapes2 import cube
#from sage.plot.plot3d.plot3d import animate
#from sage.arith.srange import srange

# Create the cube
my_cube = cube((0, 0, 0), size=2)

# Animation using f3d and make_frame3d
frames = []
for angle in srange(0, 2*pi, 0.1):
    rotated_cube = my_cube.rotate((1, 1, 0), angle)
    # Apply the f3d function to each frame
    frame = f3d(rotated_cube, grid=(10, 10, 10), colors=['white', 'white'], alpha=0.6, show_axes=True)
    frames.append(frame)

# Create an animation from the frames
animation = animate(frames)
animation.show()  # Display the animation
animation.save("cube_with_f3d.gif")  # Save as a GIF
